In [ ]:
code_dict = {
"007301":"半导体",
"012832":"新能源",
"007883":"医药",
"012700":"证券",
"012043":"酒",
"009180":"消费",
"012323":"医疗",

"007339":"沪深300",
"014345":"中证500",
"001593":"创业",

"009052":"中证红利",

"008702":"黄金"
}

In [ ]:
import akshare as ak
import pandas as pd
from datetime import datetime
import os

# ===== 日期文件名 =====
today = datetime.today().strftime("%Y-%m-%d")

os.makedirs("data", exist_ok=True)

csv_path = f"data/fund_nav_{today}.csv"

# ===== 如果已存在则跳过 =====
if os.path.exists(csv_path):
    print("Today's data already exists, skip download:")
    print(csv_path)
else:
    
    print("Start downloading fund data...")
    
    data_list = []
    
    for code, name in code_dict.items():
    
        print("downloading:", name)
    
        df = ak.fund_open_fund_info_em(
            symbol=code,
            indicator="单位净值走势"
        )
    
        df = df.rename(columns={
            "净值日期": "date",
            "单位净值": "nav"
        })
    
        df["fund"] = name
        df["code"] = code
    
        data_list.append(df[["date", "code", "fund", "nav"]])
    
    
    # ===== 合并数据 =====
    data = pd.concat(data_list)
    
    data["date"] = pd.to_datetime(data["date"]).dt.date
    
    data = data.sort_values(["date", "code"]).reset_index(drop=True)
    
    print(data.head())
    
    
    # ===== 保存 =====
    data.to_csv(csv_path, index=False)
    
    print("Saved to:", csv_path)

In [ ]:
data["return"] = data.groupby("code")["nav"].pct_change()

In [ ]:
print(type(data))
print(data.head())
print(data.dtypes)
print(data[data["code"] == "007301"].head())

In [ ]:
import pandas as pd
import numpy as np
import vectorbt as vbt

# =========================
# 1) 读取数据
# =========================
data = pd.read_csv("data/fund_nav_2026-03-16.csv", dtype={"code": str})
data["date"] = pd.to_datetime(data["date"])

# 可选：只保留你关心的基金池
fund_list = [
    # "007301",  # 半导体
    "012832",  # 新能源
    "007883",  # 医药
    "012700",  # 证券
    "012043",  # 酒
    "009180",  # 消费
    "012323",  # 医疗
    "007339",  # 沪深300
    "014345",  # 中证500
    "001593",  # 创业
    "009052",  # 中证红利
    "008702",  # 黄金
]

data = data[data["code"].isin(fund_list)].copy()

# =========================
# 2) 转成多基金宽表
# =========================
price_df = (
    data.pivot(index="date", columns="code", values="nav")
    .sort_index()
)

price_df = price_df.dropna(how="all")

# 每只基金的起始和结束日期
start_dates = price_df.apply(lambda s: s.first_valid_index())
end_dates = price_df.apply(lambda s: s.last_valid_index())

common_start = start_dates.max()
common_end = end_dates.min()

print("各基金公共起点:", common_start)
print("各基金公共终点:", common_end)

# 裁剪到公共区间
price_df_aligned = price_df.loc[common_start:common_end].copy()

# 再去掉区间内仍有缺失的日期
price_df_aligned = price_df_aligned.dropna(how="any")

print(price_df_aligned.head())
print(price_df_aligned.shape)


def build_feature_dict(price_df):
    feature_dict = {}

    # ===== 1) 基础收益类 =====
    ret_1 = price_df.pct_change(1)
    ret_5 = price_df.pct_change(5)
    ret_10 = price_df.pct_change(10)
    ret_20 = price_df.pct_change(20)
    ret_60 = price_df.pct_change(60)

    feature_dict["ret_1"] = ret_1
    feature_dict["ret_5"] = ret_5
    feature_dict["ret_10"] = ret_10
    feature_dict["ret_20"] = ret_20
    feature_dict["ret_60"] = ret_60

    # ===== 2) 动量类 =====
    momentum_5 = price_df / price_df.shift(5) - 1
    momentum_20 = price_df / price_df.shift(20) - 1
    momentum_60 = price_df / price_df.shift(60) - 1
    momentum_120 = price_df / price_df.shift(120) - 1

    feature_dict["momentum_5"] = momentum_5
    feature_dict["momentum_20"] = momentum_20
    feature_dict["momentum_60"] = momentum_60
    feature_dict["momentum_120"] = momentum_120

    # ===== 3) 波动率类 =====
    daily_ret = price_df.pct_change()

    vol_5 = daily_ret.rolling(5).std()
    vol_10 = daily_ret.rolling(10).std()
    vol_20 = daily_ret.rolling(20).std()
    vol_60 = daily_ret.rolling(60).std()

    feature_dict["vol_5"] = vol_5
    feature_dict["vol_10"] = vol_10
    feature_dict["vol_20"] = vol_20
    feature_dict["vol_60"] = vol_60

    # ===== 4) 均线/趋势类 =====
    ma_5 = price_df.rolling(5).mean()
    ma_10 = price_df.rolling(10).mean()
    ma_20 = price_df.rolling(20).mean()
    ma_60 = price_df.rolling(60).mean()
    ma_120 = price_df.rolling(120).mean()

    feature_dict["ma_ratio_5"] = price_df / ma_5 - 1
    feature_dict["ma_ratio_10"] = price_df / ma_10 - 1
    feature_dict["ma_ratio_20"] = price_df / ma_20 - 1
    feature_dict["ma_ratio_60"] = price_df / ma_60 - 1
    feature_dict["ma_ratio_120"] = price_df / ma_120 - 1

    feature_dict["ma_cross_5_20"] = ma_5 / ma_20 - 1
    feature_dict["ma_cross_10_60"] = ma_10 / ma_60 - 1
    feature_dict["ma_cross_20_120"] = ma_20 / ma_120 - 1

    # ===== 5) 回撤/位置类 =====
    rolling_max_20 = price_df.rolling(20).max()
    rolling_max_60 = price_df.rolling(60).max()
    rolling_max_120 = price_df.rolling(120).max()

    drawdown_20 = price_df / rolling_max_20 - 1
    drawdown_60 = price_df / rolling_max_60 - 1
    drawdown_120 = price_df / rolling_max_120 - 1

    feature_dict["drawdown_20"] = drawdown_20
    feature_dict["drawdown_60"] = drawdown_60
    feature_dict["drawdown_120"] = drawdown_120

    rolling_min_20 = price_df.rolling(20).min()
    rolling_min_60 = price_df.rolling(60).min()

    feature_dict["position_20"] = (price_df - rolling_min_20) / (rolling_max_20 - rolling_min_20)
    feature_dict["position_60"] = (price_df - rolling_min_60) / (rolling_max_60 - rolling_min_60)

    return feature_dict

feature_dict = build_feature_dict(price_df_aligned)

print(feature_dict.keys())
print(feature_dict["momentum_20"].head())

# =========================
# 3) 定义策略函数
# =========================
def strategy_buy_and_hold(price_df, **params):
    """第一天买入，之后一直持有"""
    entries = pd.DataFrame(False, index=price_df.index, columns=price_df.columns)
    exits = pd.DataFrame(False, index=price_df.index, columns=price_df.columns)

    if len(price_df) > 0:
        entries.iloc[0, :] = True

    return entries.astype(bool), exits.astype(bool)


def strategy_ma_cross(price_df, fast=5, slow=20, t_plus_one=False, **params):
    """多基金双均线策略"""
    fast_ma = price_df.rolling(fast).mean()
    slow_ma = price_df.rolling(slow).mean()

    entry_raw = fast_ma > slow_ma
    exit_raw = fast_ma < slow_ma

    prev_entry = entry_raw.shift(1).fillna(False).astype(bool)
    prev_exit = exit_raw.shift(1).fillna(False).astype(bool)

    entries = entry_raw & (~prev_entry)
    exits = exit_raw & (~prev_exit)

    if t_plus_one:
        entries = entries.shift(1).fillna(False).astype(bool)
        exits = exits.shift(1).fillna(False).astype(bool)

    return entries.astype(bool), exits.astype(bool)


def strategy_momentum(price_df, window=20, t_plus_one=False, **params):
    """多基金简单动量策略：过去window天收益率>0则持有"""
    momentum = price_df.pct_change(window)

    entry_raw = momentum > 0
    exit_raw = momentum <= 0

    prev_entry = entry_raw.shift(1).fillna(False).astype(bool)
    prev_exit = exit_raw.shift(1).fillna(False).astype(bool)

    entries = entry_raw & (~prev_entry)
    exits = exit_raw & (~prev_exit)

    if t_plus_one:
        entries = entries.shift(1).fillna(False).astype(bool)
        exits = exits.shift(1).fillna(False).astype(bool)

    return entries.astype(bool), exits.astype(bool)


# =========================
# 4) 策略注册表
# =========================
strategy_dict = {
    "buy_and_hold": strategy_buy_and_hold,
    "ma_cross": strategy_ma_cross,
    "momentum": strategy_momentum,
}


# =========================
# 5) 统一回测函数
# =========================
def run_backtest(price_df, strategy_name, strategy_params=None, init_cash=10000, fees=0.0):
    if strategy_params is None:
        strategy_params = {}

    strategy_func = strategy_dict[strategy_name]
    entries, exits = strategy_func(price_df, **strategy_params)

    pf = vbt.Portfolio.from_signals(
        close=price_df,
        entries=entries,
        exits=exits,
        init_cash=init_cash,
        fees=fees,
        slippage=0.0,
        freq="1D"
    )

    return pf, entries, exits


# =========================
# 6) 选择策略并运行
# =========================
strategy_name = "ma_cross"
strategy_params = {
    "fast": 5,
    "slow": 20,
    "t_plus_one": True
}

pf, entries, exits = run_backtest(
    price_df=price_df,
    strategy_name=strategy_name,
    strategy_params=strategy_params,
    init_cash=10000,
    fees=0.0
)


# =========================
# 7) 输出结果
# =========================
print("当前策略:", strategy_name)
print("策略参数:", strategy_params)

# 整体统计
print(pf.stats())

# 每个基金单独统计
print(pf.stats(group_by=False))


# =========================
# 8) 查看净值曲线
# =========================
pf.value().plot()